In [2]:
import re
import numpy as np
import pandas as pd
import copy
import ast
import csv

In [3]:
with open('input_sort.txt', 'r') as file:
    data_raw = file.read()
    
## Class
atom = {'node_w': ['name', 'V', 'W', 'T'],
           'edge_w': ['name', 'V' , "V'", 'W', 'T']
        }

columnAdded = {'node_w': {'position': 2,
                          'name': "V'"
                          }
               }

termSort =['T', 'V']
class SortAtom:
    def __init__(self, data, atom:dict, columnAdded, termSort=termSort) -> None:
        self.atomData = re.sub(' ', '\n', data)
        self.termSort = termSort
        self.atom = atom
        self.columnAdded = columnAdded
        
    # pull atom by its name from answer set
    def GetAtom(self, name):
        atomList = re.findall(name+'.*', self.atomData)
        return atomList

    # transfer to list
    def ConvertToDataFrame(self, atomName):
        atomList = self.GetAtom(atomName)
        atomListInt = [re.sub("dl\(", "", i) for i in atomList]
        atomListInt = [re.sub("\(", ",", i) for i in atomListInt]
        atomListInt = [re.sub("\)", "", i) for i in atomListInt]
        atomListInt = [re.sub(",,", ",", i) for i in atomListInt]
        atomListInt = [i.split(",") for i in atomListInt]
        atomListInt = [[int(i) if i.isdigit() else i for i in j] for j in atomListInt]
        df = pd.DataFrame(atomListInt, columns=self.atom[atomName])
        return df
    # add NaN column and create dataframe
    def RegularizeColumn(self, atomName):
        atomDf = self.ConvertToDataFrame(atomName)
        loc = self.columnAdded[atomName]['position']
        columnName= self.columnAdded[atomName]['name']
        atomDf.insert(loc, columnName, np.nan)
        return atomDf
    # sort
    def SortAll(self):
        df = pd.DataFrame()
        for atom_name in self.atom.keys():
            if atom_name in self.columnAdded.keys():
                new_df = self.RegularizeColumn(atom_name)
            else: 
                new_df = self.ConvertToDataFrame(atom_name)
            # Merge the new DataFrame with the existing one
            df = pd.concat([df, new_df], axis=0)
            df_sorted = df.sort_values(by=self.termSort)   
        return df_sorted
        

a = SortAtom(data_raw, atom, columnAdded)
a.SortAll().to_csv('output.txt', index=False, sep ='\t')


In [7]:
#AIAA sort dl
# dl(arrival(1,2,2),59)
# dl(departure(1,1,2),50)
with open('input_sort.txt', 'r') as file:
    data_raw = file.read()
    
## Class
atom = {'arrival': ['name', 'D', 'V', 'TS', 'TP'],
        'departure': ['name', 'D', 'V', 'TS', 'TP']
        }

columnAdded = {
               }

termSort =['TP', 'TS']
class SortAtom:
    def __init__(self, data, atom:dict, columnAdded, termSort=termSort) -> None:
        self.atomData = re.sub(' ', '\n', data)
        self.termSort = termSort
        self.atom = atom
        self.columnAdded = columnAdded
        
    # pull atom by its name from answer set
    def GetAtom(self, name):
        atomList = re.findall(name+'.*', self.atomData)
        return atomList

    # transfer to list
    def ConvertToDataFrame(self, atomName):
        atomList = self.GetAtom(atomName)
        atomListInt = [re.sub("dl\(", "", i) for i in atomList]
        atomListInt = [re.sub("\(", ",", i) for i in atomListInt]
        atomListInt = [re.sub("\)", "", i) for i in atomListInt]
        atomListInt = [re.sub(",,", ",", i) for i in atomListInt]
        atomListInt = [i.split(",") for i in atomListInt]
        atomListInt = [[int(i) if i.isdigit() else i for i in j] for j in atomListInt]
        df = pd.DataFrame(atomListInt, columns=self.atom[atomName])
        return df
    # add NaN column and create dataframe
    def RegularizeColumn(self, atomName):
        atomDf = self.ConvertToDataFrame(atomName)
        loc = self.columnAdded[atomName]['position']
        columnName= self.columnAdded[atomName]['name']
        atomDf.insert(loc, columnName, np.nan)
        return atomDf
    # sort
    def SortAll(self):
        df = pd.DataFrame()
        for atom_name in self.atom.keys():
            if atom_name in self.columnAdded.keys():
                new_df = self.RegularizeColumn(atom_name)
            else: 
                new_df = self.ConvertToDataFrame(atom_name)
            # Merge the new DataFrame with the existing one
            df = pd.concat([df, new_df], axis=0)
            df_sorted = df.sort_values(by=self.termSort)   
        return df_sorted
    def FormatAndWriteToFile(self, filename):
        with open(filename, 'w') as file:
            for index, row in self.SortAll().iterrows():
                # Determine the atom type (arrival or departure)
                atom_type = row['name']
                # Format the row according to the specified format
                formatted_row = f"dl({atom_type}({row['D']}, {row['V']}, {row['TS']}), {row['TP']})"
                # Write the formatted row to the file
                file.write(formatted_row + '\n')        

a = SortAtom(data_raw, atom, columnAdded)
a.SortAll().to_csv('output.txt', index=False, sep ='\t')
a.FormatAndWriteToFile('output_formatted.txt')
